# Практика 4. Наследование и исключения

Теория: блоки 8–9.

Девять задач на порядок разрешения имён и на управление ошибками.

**Как устроено.** Условие → ячейка → проверка. Ячейки, которые надо писать самому, помечены
комментарием в первой строке. Если что-то пошло не так — перезапусти ядро и выполни
служебную ячейку заново.

In [ ]:
# Служебное. Запусти эту ячейку один раз перед задачами.
def check(name, fn):
    """Запускает проверку и печатает результат."""
    try:
        fn()
        print("✓", name)
    except AssertionError as e:
        print("✗", name, "—", e)
    except Exception as e:
        print("✗", name, "—", type(e).__name__, e)


def eq(got, want):
    assert got == want, f"получено {got!r}, ожидалось {want!r}"


def expect(exc, fn):
    try:
        fn()
    except exc:
        return
    raise AssertionError(f"ожидалось {exc.__name__}")

## Задачи

### Задача 1. Порядок разрешения имён

Ромб: `B` и `C` наследуют `A`, `D` наследует обоих. Предскажи список имён в `__mro__`.

In [ ]:
# Ожидаемый вывод:
# ['D', 'B', 'C', 'A', 'object']
class A:
    pass


class B(A):
    pass


class C(A):
    pass


class D(B, C):
    pass


print([k.__name__ for k in D.__mro__])

### Задача 2. Куда приведёт `super()`

`B` и `C` не знают друг о друге, но `D` наследует обоих. Что напечатает вызов?

In [ ]:
# Ожидаемый вывод:
# DBCA
class A:
    def m(self):
        return "A"


class B(A):
    def m(self):
        return "B" + super().m()


class C(A):
    def m(self):
        return "C" + super().m()


class D(B, C):
    def m(self):
        return "D" + super().m()


print(D().m())

### Задача 3. `update` мимо переопределения

Класс наследует `dict` и перехватывает присваивание по ключу. Сколько раз сработает
перехват?

In [ ]:
# Ожидаемый вывод:
# set a
# ['a', 'b']
class D(dict):
    def __setitem__(self, key, value):
        print("set", key)
        super().__setitem__(key, value)


d = D()
d["a"] = 1
d.update({"b": 2})
print(sorted(d))

### Задача 4. Наследник, который забыл про базу

`Child()` не получает атрибут `ready`, потому что инициализатор базового класса
не вызван. Починить, не убирая собственный `__init__`.

In [ ]:
class Base:
    def __init__(self):
        self.ready = True


class Child(Base):
    def __init__(self):
        super().__init__()
        self.name = "child"

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
c = Child()
check("базовый инициализатор выполнен", lambda: eq(c.ready, True))
check("свой атрибут на месте", lambda: eq(c.name, "child"))

### Задача 5. Проверка по строке вместо полиморфизма

Функция различает фигуры по полю `"type"` и падает на всём новом. Переписать на классы:
`Shape` с абстрактным `area`, реализации `Square(side)` и `Circle(radius)`. `Shape()` должна
быть невозможна.

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        ...


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14 * self.radius ** 2

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
check("квадрат", lambda: eq(Square(2).area(), 4))
check("круг", lambda: eq(Circle(1).area(), 3.14))
check("оба — Shape", lambda: eq(isinstance(Square(1), Shape), True))
check("Shape не создаётся", lambda: expect(TypeError, lambda: Shape()))

### Задача 6. Хранилище с контрактом

**Контракт.**

- `StorageError(Exception)` — базовое исключение хранилища.
- `NotFound(StorageError)` — ключа нет.
- `Storage(ABC)` с абстрактными `save(key, value)` и `load(key)`.
- `MemoryStorage(Storage)` — держит данные в словаре.
- `load` неизвестного ключа бросает `NotFound`.
- Создать экземпляр `Storage` нельзя.

Имена даны, тела — твои.

In [ ]:
from abc import ABC, abstractmethod


class StorageError(Exception):
    pass


class NotFound(StorageError):
    pass


class Storage(ABC):
    @abstractmethod
    def save(self, key, value):
        ...

    @abstractmethod
    def load(self, key):
        ...


class MemoryStorage(Storage):
    def __init__(self):
        self.data = {}

    def save(self, key, value):
        self.data[key] = value

    def load(self, key):
        if key not in self.data:
            raise NotFound(key)
        return self.data[key]

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
check("абстрактный не создаётся", lambda: expect(TypeError, lambda: Storage()))
s = MemoryStorage()
s.save("a", 1)
check("чтение", lambda: eq(s.load("a"), 1))
check("неизвестный ключ", lambda: expect(NotFound, lambda: s.load("b")))
check("NotFound — наследник StorageError", lambda: eq(issubclass(NotFound, StorageError), True))

### Задача 7. Менеджер контекста, который глотает ошибки

**Контракт.** `Suppress(*exceptions)`:

- внутри `with` указанные исключения подавляются;
- не указанные — выходят наружу;
- подавленное исключение доступно как `s.suppressed` (до срабатывания — `None`).

Сигнатура дана, тело — твоё.

In [ ]:
class Suppress:
    def __init__(self, *exceptions):
        self.exceptions = exceptions
        self.suppressed = None

    def __enter__(self):
        self.suppressed = None
        return self

    def __exit__(self, exc_type, exc, traceback):
        if isinstance(exc, self.exceptions):
            self.suppressed = exc
            return True
        return False

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
s = Suppress(ValueError)
with s:
    raise ValueError("поймано")
check("исключение подавлено", lambda: eq(isinstance(s.suppressed, ValueError), True))


def other():
    with Suppress(ValueError):
        raise KeyError("наружу")


check("чужое проходит", lambda: expect(KeyError, other))

### Задача 8. Куда делось исключение

В задаче 7 исключение внутри `with` исчезло: блок выполнился, программа пошла дальше.
Объясни, что именно вернул `__exit__` и почему этого достаточно.

Сформулируй ответ своими словами, без кода и без запуска.

**Ответ.** При исключении `with` вызывает `__exit__`, передавая ему тип, объект исключения и traceback. Метод вернул `True`, то есть сообщил интерпретатору подавить исключение. Если вернуть `False` или `None`, исключение продолжит распространяться наружу.

### Задача 9. `return` внутри `finally`

Функция бросает исключение, но в `finally` стоит `return`. Что напечатается?

In [ ]:
# Ожидаемый вывод:
# из finally
def f():
    try:
        raise ValueError("потерялось")
    finally:
        return "из finally"


print(f())